# 02 · Data Cleaning
Runs the reproducible pipeline in `src/data/clean.py` (`python -m src.data.clean`).
**No rows are deleted.** Each anomaly becomes a `dq_*` flag plus a logged decision, so every analysis can include or
exclude flagged records explicitly.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

pd.set_option("display.max_columns", 60, "display.width", 200, "display.float_format", "{:,.4f}".format)

from src.data import clean
from src.utils.io import load_all
out = clean.run()
log = out["quality_log"]
print(f"{len(log)} checks run, {int((log.n_affected > 0).sum())} with findings")

69 checks run, 19 with findings


## Checks with findings and the decision taken

In [2]:
log[log.n_affected > 0].reset_index(drop=True)

,table,check,n_affected,n_rows,pct_affected,decision
0,accounts,seats IQR outliers,25,500,5.0000,kept: right-skewed seat counts are plausible f...
1,subscriptions,end_date == start_date (0-day subscription),13,5000,0.2600,kept + flagged: same-day cancellation is a gen...
2,subscriptions,upgrade_flag AND downgrade_flag both TRUE,23,5000,0.4600,kept + flagged: possible if plan moved both wa...
3,subscriptions,mrr_amount IQR outliers,471,5000,9.4200,"kept: explained by seats x Enterprise price, n..."
4,feature_usage,usage_id shared by different records,42,25000,0.1700,kept all rows (records differ in every attribu...
5,feature_usage,rows sharing subscription_id + usage_date + fe...,6,25000,0.0200,kept + flagged: multiple sessions of one featu...
6,feature_usage,usage_count == 0,2,25000,0.0100,kept + flagged (also have 0 duration: consiste...
7,feature_usage,usage_duration_secs IQR outliers,145,25000,0.5800,"kept: max 12,696 s (3.5 h) is plausible"
8,feature_usage,usage_date before its subscription start_date,19142,25000,76.5700,kept + flagged: systemic (generator did not al...
9,feature_usage,usage_date after its subscription end_date,290,25000,1.1600,kept + flagged


## Checks that passed (nothing found)

In [3]:
log[log.n_affected == 0][["table", "check"]].reset_index(drop=True)

,table,check
0,accounts,full-row duplicates
1,accounts,duplicate account_id
2,accounts,unparseable signup_date
3,accounts,signup_date after snapshot 2024-12-31
4,accounts,industry outside expected categories
5,accounts,country outside expected categories
6,accounts,referral_source outside expected categories
7,accounts,plan_tier outside expected categories
8,accounts,seats <= 0
9,subscriptions,full-row duplicates


## Processed tables (reloaded with dtypes)

In [4]:
t = load_all()
pd.DataFrame({n: {"rows": len(df), "columns": df.shape[1], "dq_flag_columns": sum(c.startswith("dq_") for c in df.columns)} for n, df in t.items()}).T

,rows,columns,dq_flag_columns
accounts,500,17,1
subscriptions,5000,24,6
feature_usage,25000,17,6
support_tickets,2000,14,3
churn_events,600,11,1


In [5]:
# Example: flags on feature_usage
fu = t["feature_usage"]
fu.filter(regex="^dq_|is_in_subscription_window").mean().rename("share_of_rows").to_frame()

,share_of_rows
dq_duplicate_usage_id,0.0017
dq_same_sub_date_feature,0.0002
dq_zero_usage,0.0001
dq_before_subscription_start,0.7657
dq_after_subscription_end,0.0116
dq_before_account_signup,0.5279
is_in_subscription_window,0.2227


In [6]:
# Account churn reconciliation columns added during cleaning
t["accounts"][["account_id", "churn_flag", "n_churn_events", "first_churn_date", "has_churn_event", "dq_churn_flag_conflict"]].head()

,account_id,churn_flag,n_churn_events,first_churn_date,has_churn_event,dq_churn_flag_conflict
0,A-2e4581,False,2,2024-11-23,True,True
1,A-43a9e3,True,0,NaT,False,True
2,A-0a282f,False,2,2024-10-06,True,True
3,A-1f0ac7,False,1,2024-11-08,True,True
4,A-ce550d,True,1,2024-12-28,True,False


## Cleaning decisions (summary)
| Issue | Decision |
|---|---|
| 21 `usage_id` values reused by different records | keep all rows, new surrogate key `usage_row_id` |
| usage before subscription start (76.6%) / before signup (52.8%) | keep + flag; use usage as lifetime behaviour only |
| tickets before signup (53.9%) | keep + flag; tickets used as lifetime support load |
| 36 tickets whose first response is slower than resolution | keep + flag; excluded from first-response metrics |
| 1 repeated churn event (same account + date) | keep + flag; excluded from event counts |
| 312 accounts where `churn_flag` disagrees with `churn_events` | keep + flag; `churn_flag` = status, events = episodes |
| 13 zero-day subscriptions, 23 with both upgrade & downgrade flags | keep + flag (plausible business cases) |
| seat / MRR outliers | keep: explained by deterministic seat pricing |
| NULL `satisfaction_score`, `feedback_text`, `end_date` | keep NULL (documented meaning); never imputed |